### 1. Library Imports
Importing all required PyTorch, TorchVision, and utility modules for training and preprocessing.

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print("Libraries imported successfully. PyTorch version:", torch.__version__)

Libraries imported successfully. PyTorch version: 2.11.0+cpu


### 2. Configuration & Hyperparameters
Setting up image dimensions, batch sizes, and paths to match our local repository structure.

In [3]:
# Paths
DATA_DIR = "./datasets/Potato"
MODEL_SAVE_PATH = "../weights/potato_model.pth"

# Hyperparameters
BATCH_SIZE = 32
IMAGE_SIZE = (128, 128)  # Optimized for FPGA memory constraints
EPOCHS = 10
LEARNING_RATE = 0.001

### 3. Data Augmentation & Loading
Applying transforms and oversampling logic to handle the healthy-class scarcity (152 healthy vs 2,000 diseased images).

In [4]:
# Augmentation pipeline to artificially enrich the healthy class during training
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset using standard ImageFolder structure
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transforms)
print("Original class folders found:", full_dataset.classes)

Original class folders found: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


### 4. Binary Mapping Logic
Mapping the 3-folder structure down into a strict binary classification problem: 
- 0 = Healthy
- 1 = Diseased (combining Early and Late Blight)

In [5]:
from torch.utils.data import random_split

def binary_target_transform(target_class_idx):
    folder_name = full_dataset.classes[target_class_idx]
    if "healthy" in folder_name.lower():
        return 0  # Healthy
    else:
        return 1  # Diseased (Early or Late Blight)

# Apply target mapping
full_dataset.target_transform = binary_target_transform

# Perform a rigorous 80/20 train/test split using seed 42 to match evaluate.py exactly
torch.manual_seed(42)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Initialize DataLoader using ONLY the training partition
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Total dataset images loaded: {len(full_dataset)}")
print(f"Training split size: {len(train_dataset)} images")
print(f"Holdout test split size (reserved for evaluate.py): {len(test_dataset)} images")

Total dataset images loaded: 2152
Training split size: 1721 images
Holdout test split size (reserved for evaluate.py): 431 images


### 5. Lightweight CNN Model Architecture
A compact convolutional network designed for smooth INT8 post-training quantization and easy hardware deployment.

In [6]:
class PotatoBinaryCNN(nn.Module):
    def __init__(self):
        super(PotatoBinaryCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        # For 128x128 input, feature map size after 2 max pools is 32x32
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # Binary output logit
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = PotatoBinaryCNN()
print(model)

PotatoBinaryCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


### 6. Loss Function & Optimizer Setup
Configuring binary cross-entropy with logits and class weighting to penalize misclassifications on the rare healthy class.

In [7]:
# Class weights to handle imbalance (approx ratio compensation)
# Diseased count (~2000), Healthy count (~152)
pos_weight = torch.tensor([2000.0 / 152.0]) 

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

### 7. Model Training Loop
Executing training across epochs and saving the trained weights for subsequent quantization and MIF export.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Training on device: {device}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss:.4f}")

# Save trained weights
os.makedirs("../weights", exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model weights successfully saved to {MODEL_SAVE_PATH}")

Training on device: cpu
Epoch [1/10], Loss: 0.6225
Epoch [2/10], Loss: 0.2678
Epoch [3/10], Loss: 0.1624
Epoch [4/10], Loss: 0.1426
Epoch [5/10], Loss: 0.1151
Epoch [6/10], Loss: 0.1249
Epoch [7/10], Loss: 0.1384
Epoch [8/10], Loss: 0.0734
Epoch [9/10], Loss: 0.0883
Epoch [10/10], Loss: 0.0729
Model weights successfully saved to ../weights/potato_model.pth
